In [17]:
!pip install mediapipe pandas seaborn scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
import cv2
import psutil
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from IPython.display import display, clear_output

model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
model_path = "hand_landmarker.task"

if not os.path.exists(model_path):
    print("Downloading MediaPipe Hand Landmarker model...")
    urllib.request.urlretrieve(model_url, model_path)
    print("Download complete!")

print("Libraries imported and model weights ready.")

Cell 1 Executed: Libraries imported and model weights ready.


feature extraction

In [19]:
def extract_landmarks_from_image(image_path, detector):
    if not os.path.exists(image_path):
        return None
    
    mp_image = mp.Image.create_from_file(image_path)
    detection_result = detector.detect(mp_image)

    
    vector = [0.0] * 63 

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]
        
        base_x, base_y, base_z = hand_landmarks[0].x, hand_landmarks[0].y, hand_landmarks[0].z
        
        max_dist = 0
        for lm in hand_landmarks:
            dist = np.sqrt((lm.x - base_x)**2 + (lm.y - base_y)**2)
            if dist > max_dist: max_dist = dist
        
        scale = max_dist if max_dist > 0 else 1

        for j, lm in enumerate(hand_landmarks):
            vector[j*3]     = (lm.x - base_x) / scale
            vector[j*3 + 1] = (lm.y - base_y) / scale
            vector[j*3 + 2] = (lm.z - base_z) / scale
                
        return vector
    return None

def build_feature_dataset(dataset_dir):
    data, labels = [], []
    base_options = mp_tasks.BaseOptions(model_asset_path='hand_landmarker.task')
    options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1, min_hand_detection_confidence=0.5)
    
    with vision.HandLandmarker.create_from_options(options) as detector:
        for label in sorted(os.listdir(dataset_dir)):
            label_dir = os.path.join(dataset_dir, label)
            if os.path.isdir(label_dir):
                print(f"[*] Processing: {label}")
                for img_file in os.listdir(label_dir):
                    features = extract_landmarks_from_image(os.path.join(label_dir, img_file), detector)
                    if features:
                        data.append(features)
                        labels.append(label.upper())

    df = pd.DataFrame(data, columns=[f"f_{i}" for i in range(63)])
    df['label'] = labels
    return df

smart diagnostic and extraction cell

In [ ]:
import os
import pandas as pd

print("Hunting for the exact dataset path dynamically...")

DATASET_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'full_dataset' in dirs:
        potential_path = os.path.join(root, 'full_dataset')
        subdirs = [d for d in os.listdir(potential_path) if os.path.isdir(os.path.join(potential_path, d))]
        if subdirs:
            DATASET_PATH = potential_path
            break

if DATASET_PATH is None:
    print("Dynamic search failed. Checking common Kaggle paths...")
    backup_path = '/kaggle/input/full_dataset/full_dataset'
    if os.path.exists(backup_path):
        DATASET_PATH = backup_path

if DATASET_PATH is None:
    print("Could not find the folder 'full_dataset'.")
    print("Available files in /kaggle/input:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
else:
    print(f" Found the dataset at: {DATASET_PATH}")
    print(f" Labels detected: {sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])}")
    print(" Starting MediaPipe feature extraction...")
    df_features = build_feature_dataset(DATASET_PATH)
    
    if not df_features.empty:
        df_features.to_csv('sign_features_63d.csv', index=False)
        print(f" Done! Processed {len(df_features)} images and saved 'sign_features_63d.csv'.")
        display(df_features.head())
    else:
        print(" Extraction finished, but no valid features were generated.")

Hunting for the exact dataset path dynamically...
Dynamic search failed. Checking common Kaggle paths...
 ERROR: Could not find the folder 'full_dataset'.
Available files in /kaggle/input:


Model definition and training

In [21]:
import joblib
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def train_mlp_model(csv_path):
    if not os.path.exists(csv_path):
        print(f"Error: '{csv_path}' not found. Run the extraction cell first.")
        return None, None

    df = pd.read_csv(csv_path)
    X = df.drop('label', axis=1).values
    y = df['label'].astype(str).values 
    
    classes = np.unique(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        batch_size=32,
        learning_rate='adaptive',
        max_iter=1000,
        warm_start=True,
        random_state=42
    )

    print("Training MLP model...")
    start_time = time.time()
    mlp.fit(X_train, y_train)
    
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"Training Complete in {time.time() - start_time:.2f} seconds!")
    print(f"Baseline Test Accuracy: {acc * 100:.2f}%")
    
    joblib.dump((mlp, classes), 'sign_language_mlp.pkl')
    print("Model saved as 'sign_language_mlp.pkl'")
    
    return mlp, classes

csv_filename = 'sign_features_63d.csv'
mlp_model, known_classes = train_mlp_model(csv_filename)

Training MLP model...
Training Complete in 59.52 seconds!
Baseline Test Accuracy: 99.35%
Model saved as 'sign_language_mlp.pkl'


utility functions and system components

In [22]:
class SignSpaceTracker:
    def __init__(self, space_timeout=1.5):
        self.space_timeout = space_timeout
        self.last_seen_time = time.time()
        self.space_triggered = False

    def update(self, hand_detected):
        current_time = time.time()
        if hand_detected:
            self.last_seen_time = current_time
            self.space_triggered = False
            return False
        elif (current_time - self.last_seen_time > self.space_timeout) and not self.space_triggered:
            self.space_triggered = True
            return True
        return False

class TranscriptManager:
    def __init__(self, stability_threshold=10):
        self.current_word = ""
        self.sentence = []
        self.prediction_buffer = [] 
        self.stability_threshold = stability_threshold

    def process_prediction(self, letter):
        self.prediction_buffer.append(letter)
        if len(self.prediction_buffer) > self.stability_threshold:
            self.prediction_buffer.pop(0)

        if len(self.prediction_buffer) == self.stability_threshold and len(set(self.prediction_buffer)) == 1:
            stable_letter = self.prediction_buffer[0]
            if not self.current_word or self.current_word[-1] != stable_letter:
                self.current_word += stable_letter
                self.prediction_buffer.clear()

    def trigger_space(self):
        if self.current_word:
            self.sentence.append(self.current_word)
            self.current_word = ""

    def get_transcript(self):
        return " ".join(self.sentence + ([self.current_word] if self.current_word else []))

class PerformanceDashboard:
    def __init__(self):
        self.prev_time = time.time()
        self.process = psutil.Process(os.getpid())
        
    def get_metrics(self):
        curr_time = time.time()
        latency = curr_time - self.prev_time
        fps = 1 / latency if latency > 0 else 0
        self.prev_time = curr_time
        
        cpu = self.process.cpu_percent()
        mem = self.process.memory_info().rss / (1024 * 1024)
        
        return int(fps), int(latency * 1000), cpu, mem

print("Utility components loaded")

Utility components loaded


In [26]:
import cv2
import time
import os
import psutil
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision
import joblib
import tkinter as tk
from tkinter import simpledialog

root = tk.Tk()
root.withdraw()

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),    
    (0, 5), (5, 6), (6, 7), (7, 8),    
    (5, 9), (9, 10), (10, 11), (11, 12), 
    (9, 13), (13, 14), (14, 15), (15, 16), 
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)
]

class TranscriptManager:
    def __init__(self, hold_duration=1.75, max_display_chars=20):
        self.current_word = ""
        self.sentence = []
        self.last_letter = None
        self.start_hold_time = 0
        self.hold_duration = hold_duration
        self.is_confirmed = False
        self.max_display_chars = max_display_chars

    def process_prediction(self, letter):
        current_time = time.time()
        if letter != self.last_letter:
            self.last_letter = letter
            self.start_hold_time = current_time
            self.is_confirmed = False
            return None
        if not self.is_confirmed and letter is not None:
            elapsed = current_time - self.start_hold_time
            if elapsed >= self.hold_duration:
                if letter.upper() == "SPACE":
                    if self.sentence or self.current_word:
                        self.sentence.append(self.current_word)
                        self.current_word = ""
                else:
                    self.current_word += letter
                self.is_confirmed = True
                return "READY"
            return elapsed
        return None

    def get_transcript(self):
        full_text = " ".join(self.sentence + ([self.current_word] if self.current_word else []))
        if len(full_text) > self.max_display_chars:
            return "..." + full_text[-self.max_display_chars:]
        return full_text

LABEL_MAP = {"2O": "2", "6O": "6", "SPACE": " "}


try:
    if 'mlp_model' in globals() and 'known_classes' in globals():
        mlp = mlp_model
        classes = known_classes
    else:
        mlp, classes = joblib.load('sign_language_mlp.pkl')
    
    mlp.warm_start = False
    n_features = mlp.n_features_in_
    all_known_classes = np.array(classes)
except Exception as e:
    raise RuntimeError(f"Model load failed: {e}")


base_options = mp_tasks.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.6
)
detector = vision.HandLandmarker.create_from_options(options)

transcript = TranscriptManager(hold_duration=1.75)
cap = cv2.VideoCapture(0)
show_skeleton = False

try:
    print(" Started. 'Q'-Quit, 'C'-Correct, 'S'-Toggle Skeleton")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        results = detector.detect(mp_image)
        hand_detected = bool(results.hand_landmarks)
        current_features = [0.0] * n_features
        
        if hand_detected:
            hand_lms = results.hand_landmarks[0]
            
            if show_skeleton:
                for connection in HAND_CONNECTIONS:
                    start_idx, end_idx = connection
                    pt1 = (int(hand_lms[start_idx].x * w), int(hand_lms[start_idx].y * h))
                    pt2 = (int(hand_lms[end_idx].x * w), int(hand_lms[end_idx].y * h))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2)
                for lm in hand_lms:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            
            bx, by, bz = hand_lms[0].x, hand_lms[0].y, hand_lms[0].z
            max_d = max([np.sqrt((l.x-bx)**2 + (l.y-by)**2) for l in hand_lms])
            scale = max_d if max_d > 0 else 1
            
            for j, l in enumerate(hand_lms):
                current_features[j*3] = (l.x - bx) / scale
                current_features[j*3+1] = (l.y - by) / scale
                current_features[j*3+2] = (l.z - bz) / scale

            
            X_live = np.array(current_features).reshape(1, -1)
            probs = mlp.predict_proba(X_live)[0]
            raw_pred = classes[np.argmax(probs)]
            confidence = np.max(probs)

            display_char = LABEL_MAP.get(raw_pred, raw_pred)
            x_min = int(min([l.x * w for l in hand_lms]))
            y_min = int(min([l.y * h for l in hand_lms]))
            
            # cv2.putText(frame, f"{display_char} ({int(confidence*100)}%)", 
            #             (x_min, y_min - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"{raw_pred} ({int(confidence*100)}%)", 
                        (x_min, y_min - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            
            status = transcript.process_prediction(display_char)
            if isinstance(status, (float, int)):
                progress_w = int((status / 1.75) * 150)
                cv2.rectangle(frame, (x_min, y_min - 35), (x_min + 150, y_min - 25), (50, 50, 50), -1)
                cv2.rectangle(frame, (x_min, y_min - 35), (x_min + progress_w, y_min - 25), (0, 255, 255), -1)

        
        cv2.rectangle(frame, (0, h - 60), (w, h), (0, 0, 0), -1)
        sk_status = "ON" if show_skeleton else "OFF"
        cv2.putText(frame, f"SKEL: {sk_status} | TEXT: {transcript.get_transcript()}", (20, h - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.imshow('SignTalk Live HUD', frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): 
            break
        elif key == ord('s'): 
            show_skeleton = not show_skeleton
        elif key == ord('c') and hand_detected:
            new_label = simpledialog.askstring("Correction", "Correct label:").strip().upper()
            if new_label and new_label in all_known_classes:
                for _ in range(2): 
                    mlp.partial_fit([current_features], [new_label], classes=all_known_classes)
                print(f" Updated for: {new_label}")

finally:
    if 'mlp' in locals():
        joblib.dump((mlp, classes), 'updated_sign_model.pkl')
    cap.release()
    cv2.destroyAllWindows()
    root.destroy()

 Started. 'Q'-Quit, 'C'-Correct, 'S'-Toggle Skeleton
